In [3]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

In [4]:
path = "/Users/ajithsreepuram/Desktop/Customer-Churn predictor/data/"
X_train = pd.read_csv(path + "X_train.csv")
X_test  = pd.read_csv(path + "X_test.csv")
Y_train = pd.read_csv(path + "Y_train.csv").squeeze()
Y_test  = pd.read_csv(path + "Y_test.csv").squeeze()
scaler  = pickle.load(open(path + "scaler.pkl", "rb"))

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Data loaded successfully. Shape:", X_train.shape)

Data loaded successfully. Shape: (5634, 34)


In [5]:
sm = SMOTE(random_state=42)
X_train_res, Y_train_res = sm.fit_resample(X_train_scaled, Y_train)
print(f"Before SMOTE — Churn: {Y_train.sum()} | No Churn: {(Y_train==0).sum()}")
print(f"After SMOTE  — Churn: {Y_train_res.sum()} | No Churn: {(Y_train_res==0).sum()}")

Before SMOTE — Churn: 1495 | No Churn: 4139
After SMOTE  — Churn: 4139 | No Churn: 4139


In [6]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train_res, Y_train_res)

Y_pred_rf = rf.predict(X_test_scaled)
Y_prob_rf  = rf.predict_proba(X_test_scaled)[:,1]

print("=== Random Forest ===")
print(classification_report(Y_test, Y_pred_rf))
print(f"ROC-AUC: {roc_auc_score(Y_test, Y_prob_rf):.4f}")

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1035
           1       0.57      0.59      0.58       374

    accuracy                           0.77      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.77      0.77      1409

ROC-AUC: 0.8254


In [7]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=3,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train_res, Y_train_res)

Y_pred_xgb = xgb.predict(X_test_scaled)
Y_prob_xgb = xgb.predict_proba(X_test_scaled)[:,1]

print("=== XGBoost ===")
print(classification_report(Y_test, Y_pred_xgb))
print(f"ROC-AUC: {roc_auc_score(Y_test, Y_prob_xgb):.4f}")

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.92      0.66      0.77      1035
           1       0.48      0.85      0.61       374

    accuracy                           0.71      1409
   macro avg       0.70      0.76      0.69      1409
weighted avg       0.81      0.71      0.73      1409

ROC-AUC: 0.8376


In [8]:
import pandas as pd
import pickle

results = {
    "Model":      ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC-AUC":    [0.8460,               0.8254,          0.8376],
    "F1-Churn":   [0.59,                 0.64,            0.61],
    "Recall-Churn":[0.53,                0.60,            0.85],
    "Precision-Churn":[0.67,             0.68,            0.48]
}

df_results = pd.DataFrame(results)
print("=== Model Comparison ===")
print(df_results.to_string(index=False))
print("\nBest ROC-AUC:     Logistic Regression (0.8460)")
print("Best Churn Recall: XGBoost (0.85) — catches most actual churners")
print("Selected Model:    XGBoost — best business value")
path = "/Users/ajithsreepuram/Desktop/Customer-Churn predictor/models/"
pickle.dump(xgb, open(path + "xgb_model.pkl", "wb"))
print("\nXGBoost model saved to models/xgb_model.pkl")

=== Model Comparison ===
              Model  ROC-AUC  F1-Churn  Recall-Churn  Precision-Churn
Logistic Regression   0.8460      0.59          0.53             0.67
      Random Forest   0.8254      0.64          0.60             0.68
            XGBoost   0.8376      0.61          0.85             0.48

Best ROC-AUC:     Logistic Regression (0.8460)
Best Churn Recall: XGBoost (0.85) — catches most actual churners
Selected Model:    XGBoost — best business value

XGBoost model saved to models/xgb_model.pkl
